In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('qoute_dataset.csv')

In [4]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
df.shape

(3038, 2)

## NLP

**Preprocessing**

In [7]:
quotes = df['quote']

In [8]:
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [12]:
import string

In [9]:
quotes = quotes.str.lower()

In [13]:
translator = str.maketrans('', '', string.punctuation)

In [14]:
quotes = quotes.apply(lambda x:x.translate(translator))

In [15]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

**Tokenization**

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [18]:
vocab_size = 10000

tok = Tokenizer(num_words=vocab_size)
tok.fit_on_texts(quotes)

In [22]:
word_index = tok.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [23]:
sequence = tok.texts_to_sequences(quotes)

In [24]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [25]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

**Vectorization**

In [31]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [36]:
len(X)

85271

In [37]:
len(y)

85271

In [38]:
#Padding

maxLen = max(len(x) for x in X)
print(maxLen)

745


In [39]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [41]:
X_padded = pad_sequences(X, maxlen=maxLen, padding='pre')
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [42]:
y = np.array(y)
y

array([ 62,  29,  19, ...,   3, 169, 101], shape=(85271,))

**One Hot Encoding**

In [43]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [44]:
y.shape

(85271,)

In [45]:
y_one_hot.shape

(85271, 10000)

## Modelling

In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [47]:
embedding_dim = 50
rnn_units = 128

**RNN Model**

In [48]:
rnn_model = Sequential()

rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxLen))

rnn_model.add(SimpleRNN(units=rnn_units))

rnn_model.add(Dense(units=vocab_size, activation='softmax'))

c:\Users\mayan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [56]:
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [50]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**LSTM Model**

In [52]:
lstm_model = Sequential()

lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxLen))

lstm_model.add(LSTM(units=rnn_units))

lstm_model.add(Dense(units=vocab_size, activation='softmax'))

c:\Users\mayan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [57]:
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [58]:
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [59]:
epochs = 10
batch_size = 128

In [62]:
# history_rnn = rnn_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size, validation_split=0.1)

In [ ]:
# rnn_model.save("rnn_model.h5")

**OR**

In [63]:
# history_lstm = lstm_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size, validation_split=0.1)

In [ ]:
# lstm_model.save("lstm_model.h5")

In [64]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [65]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [66]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# def predictor(model, tok, text, maxLen):
#     text = text.lower()

#     seq = tok.texts_to_sequences([text])[0]
#     seq = pad_sequences([seq], maxlen=maxLen, padding='pre')

#     pred = model.predict(seq, verbose=0)
#     pred_index = np.argmax(pred)
#     return index_to_word[pred_index]

In [ ]:
# seed_text = "Here is"
# next_word = predictor(lstm_model, tok, seed_text, maxLen)
# print(next_word)

wolfy


In [ ]:
# def generateText(model, tok, seed_text, maxLen, n_words):
#     text = seed_text
#     for _ in range(n_words):
#         next_word = predictor(model, tok, seed_text, maxLen)
#         if next_word == '':
#             break
#         seed_text += " " + next_word
#     return seed_text

In [81]:
# seed_text = 'My name'
# generateText = generateText(lstm_model, tok, seed_text, maxLen, 10)
# print(generateText)

In [80]:
# import pickle
# with open('tokenizer.pkl', 'wb') as file:
#     pickle.dump(tok, file)

In [ ]:
# with open("maxLen.pkl", 'wb') as file:
#     pickle.dump(maxLen, file)